In [ ]:
import pypsa
import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
        "plot_global_supply",
        scenario="default", # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
        sort=True,
        demand="", #"__unreserved" or "" if reserved. Affects only the supply curve from single curves, not from the trade model
        wacc="regional",
        cost_year="2050",
        interone="hbi",
        trade_chain="newre1206_2050"
)

In [ ]:
scenario = snakemake.wildcards.scenario
sort = snakemake.wildcards.sort
demand = snakemake.wildcards.demand

In [ ]:
process = "hbi"  # or "eaf" # DISCLAIMER: Steel only supply curve "steel" from model not supported yet
xlim = None
ylim = 1000
add_iron_ore_cost = True
plot_optimal_curve = False # Uses the optimal supply curve from the trade model, which may differ from the single curves due to the trade-offs between different producers. If False, the single curves are plotted, which may not reflect the optimal supply curve from the trade model.

### Description

This notebook plots the global supply curve with different approaches. Once from the supply curves, and once from the trade model. The results should be identical, unless the parameter `add_iron_ore_cost` is `True`, because the trade model uses iron ore costs from trade results whereas the supply curves add a generic cost premium

### Get steel demand

In [ ]:
steel_demand = pd.read_csv(snakemake.input.steel_demand, index_col="region")

### Steel supply curve all regions (from model)

In [ ]:
if process == "hbi":
    trade_model = snakemake.input.trade_network
else:
    print("WARNING: Steel only supply curve 'steel' from model not supported yet")

### Read trade model


In [ ]:
n = pypsa.Network(trade_model)

In [ ]:
# Iron ore cost per ton of steel (w/o transport)
iron_ore_to_steel = snakemake.config["iron_ore"]["ore_to_steel_ratio"]
iron_ore_cost_notransport = (
    n.generators[n.generators.carrier == "iron_ore"].marginal_cost.mean()
    * iron_ore_to_steel
)
print(f"Iron ore cost (w/o transport): {iron_ore_cost_notransport:.2f} €/t_steel")

In [ ]:
# Select process
interone = n.links[n.links.carrier == "hbi"]

In [ ]:
interone.loc[:, "region"] = interone["bus1"].str.replace("_hbi$", "", regex=True)

In [ ]:
# Marginal costs refer to bus0 (which is iron ore). To get the cost of steel (bus1), the process efficiency needs to be taken into account
iron_ore_to_steel = 1 / interone.efficiency.mean()

interone.loc[:, "cost in €/t_steel"] = interone.marginal_cost.values * iron_ore_to_steel
interone.loc[:, "cost in €/t_steel incl. iron ore"] = (
    interone["cost in €/t_steel"] + iron_ore_cost_notransport
)

In [ ]:
# Add quantity
if plot_optimal_curve:
    interone.loc[:, "quantity in Mt_steel"] = (
        interone.p_nom_opt.values / iron_ore_to_steel / 1e6
    )
else:
    interone.loc[:, "quantity in Mt_steel"] = (
        interone.p_nom_max.values / iron_ore_to_steel / 1e6
    )

In [ ]:
# Sort by cost for supply curve
interone_sorted = interone.sort_values("cost in €/t_steel"+(" incl. iron ore" if add_iron_ore_cost else "")).reset_index(
    drop=True
)

# Assign a color to each region
region_list = interone_sorted["region"].unique()
region_colors = snakemake.config["colors"]

cum_quantity_line = interone_sorted["quantity in Mt_steel"].cumsum()
cum_cost_line = interone_sorted["cost in €/t_steel"+(" incl. iron ore" if add_iron_ore_cost else "")] 

# Prepare for stacked area plot: for each link, plot a bar at its cost, colored by region
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in interone_sorted.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row["quantity in Mt_steel"])
    bar_costs.append(row["cost in €/t_steel"+(" incl. iron ore" if add_iron_ore_cost else "")])
    bar_colors.append(region_colors[row["region"]])
    cum_quantity += row["quantity in Mt_steel"]

In [ ]:
def plot_supply_curve(
    bar_lefts, bar_widths, bar_costs, bar_colors, region_colors, region_list, xlim=2500, ylim=1500
):

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(
        bar_lefts,
        bar_costs,
        width=bar_widths,
        color=bar_colors,
        align="edge",
        edgecolor="none",
        alpha=0.8,
    )
    # Add the classic supply curve line

    ax.step(
        cum_quantity_line,
        cum_cost_line,
        where="pre",
        color="black",
        linewidth=1.2,
        label="Supply curve",
    )

    # Add vertical line of current demand and REMIND demand
    current_demand = 2000  # Mt steel
    ax.axvline(
        current_demand, color="black", linestyle="--", label="Current demand (2020)", alpha=0.6,
    )
    ax.annotate(
        "Grey steel \ndemand 2025",
        xy=(current_demand, 50),
        xytext=(current_demand + 50, 750),
        rotation=90,
        color="black",
        alpha=0.6,
    )
    remind_demand = steel_demand["SteelDemand_DRI_Mt"].sum()  # Mt steel, REMIND global steel long-term hydrogen: 700-750 Mt
    ax.axvline(
        remind_demand, color="black", linestyle="--", label="REMIND 2050 demand", alpha=0.6,
    )
    ax.annotate(
        "Green steel \ndemand 2050",
        xy=(remind_demand, 50),
        xytext=(remind_demand + 50, 750),
        rotation=90,
        color="black",
        alpha=0.6,
    )

    ax.set_xlabel("Cumulative quantity (Mt)")
    ax.set_ylabel("Cost of HBI in €/t$_{hbi}$")
    # ax.set_title(f"Combined {process} supply curve")
    ax.set_ylim(0, ylim)
    ax.set_xlim(0, sum(bar_widths))
    ax.set_xlim(0, xlim)
    ax.grid(axis="y", alpha=0.4, zorder=0)
    # Legend for regions
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=region_colors[reg]) for reg in region_list
    ]
    handles.append(
        plt.Line2D([0], [0], color="black", linewidth=1.2, label="Supply curve")
    )
    labels = list(region_list) + ["Supply curve"]
    ax.legend(
        handles, labels, title="Region", bbox_to_anchor=(1.05, 1), loc="upper left"
    )
    plt.tight_layout()
    plt.savefig(snakemake.output.network_curve)
    plt.savefig(snakemake.output.network_curve_png, dpi=300)
    plt.show()

    return

In [ ]:
plot_supply_curve(
    bar_lefts=bar_lefts,
    bar_widths=bar_widths,
    bar_costs=bar_costs,
    bar_colors=bar_colors,
    region_colors=region_colors,
    region_list=region_list,
    xlim=xlim,
    ylim=ylim,
)

### Steel supply curve (from single curves)

In [ ]:
sort_over_all = False  # "True" to sort over all regions, "False" to sort within each region
sort = "cost_global"  # "cost_global" # "cost_global", "cost_average", "False", or the variable sort (obtained from snakemake workflow)

In [ ]:
ironorecost = 97 * 1.59
regions = snakemake.config["regions"].keys()

In [ ]:
def sort_by_average(df_all):
    # Drop rows where either cost or demand is NaN before calculating weighted average
    df_clean = df_all.dropna(subset=[ "lcox [EUR/t]", "demand [t]"])
    # Get weighted average cost per region
    df_clean["weighted_cost"] = df_clean["lcox [EUR/t]"] * df_clean["demand [t]"]
    average_cost_per_region = (
        df_clean.groupby("region")["weighted_cost"].sum()
        / df_clean.groupby("region")["demand [t]"].sum()
    )
    # Sort df_all by average_cost_per_region, then by cox within each region
    df_all["avg_region_cost"] = df_all["region"].map(average_cost_per_region)
    df_all = df_all.sort_values(["avg_region_cost", "lcox [EUR/t]"]).drop(
        columns="avg_region_cost"
    )
    return df_all

In [ ]:
df_all = pd.DataFrame()

# Select input files directly from snakemake (avoids hardcoded paths)
input_files = snakemake.input.supply_curves_interone

for fn in input_files:
    # Extract region from filename pattern: {region}_{process}.csv
    region = fn.split("/")[-1].split("_marginal_cost")[0]

    df = pd.read_csv(fn)
    df["region"] = region

    # Calculate the difference in demand to get the quantity supplied at each cost step
    diff = df["demand [t]"].diff()
    diff[0] = diff[1]  # set first value to second to avoid NaN
    df["demand [t]"] = diff

    df_all = pd.concat([df_all, df], ignore_index=True)

# Remove rows where the lcox [EUR/t] value is NaN (e.g. due to missing data for some regions)
df_all = df_all.dropna(subset=["lcox [EUR/t]"])


if sort_over_all:
    df_all = df_all.sort_values("lcox [EUR/t]").reset_index(drop=True)

if sort == False:
    pass
elif sort == "cost_global":
    df_all = df_all.sort_values("lcox [EUR/t]").reset_index(drop=True)
elif sort == "cost_average":
    df_all = sort_by_average(df_all)
else:
    ValueError(f"value for sort is {sort} and not valid")

if add_iron_ore_cost:
    df_all["lcox [EUR/t]"] += ironorecost

In [ ]:
cum_quantity_line = df_all["demand [t]"].cumsum() / 1e6  # Mt steel
cum_cost_line = df_all["lcox [EUR/t]"]

# Prepare for stacked area plot based on df_all
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in df_all.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row["demand [t]"] / 1e6)  # Mt steel
    bar_costs.append(row["lcox [EUR/t]"])
    bar_colors.append(region_colors[row["region"]])
    cum_quantity += row["demand [t]"] / 1e6  # Mt steel

In [ ]:
plot_supply_curve(
    bar_lefts=bar_lefts,
    bar_widths=bar_widths,
    bar_costs=bar_costs,
    bar_colors=bar_colors,
    region_colors=region_colors,
    region_list=region_list,
    xlim=xlim,
    ylim=ylim,
)